##### Copyright 2024 Google LLC.

In [ ]:
#@title Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
# https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# Explore vision capabilities with the Gemini API

<table class="tfo-notebook-buttons" align="left">
  <td>
    <a target="_blank" href="https://ai.google.dev/gemini-api/docs/vision"><img src="https://ai.google.dev/static/site-assets/images/docs/notebook-site-button.png" height="32" width="32" />View on ai.google.dev</a>
  <td>
    <a target="_blank" href="https://colab.research.google.com/github/google/generative-ai-docs/blob/main/site/en/gemini-api/docs/vision.ipynb"><img src="https://www.tensorflow.org/images/colab_logo_32px.png" />Run in Google Colab</a>
  </td>
  <td>
    <a target="_blank" href="https://github.com/google/generative-ai-docs/blob/main/site/en/gemini-api/docs/vision.ipynb"><img src="https://www.tensorflow.org/images/GitHub-Mark-32px.png" />View source on GitHub</a>
  </td>
</table>

The Gemini API is able to process images and videos, enabling a multitude of
 exciting developer use cases. Some of Gemini's vision capabilities include
 the ability to:

*   Caption and answer questions about images
*   Transcribe and reason over PDFs, including long documents up to 2 million token context window
*   Describe, segment, and extract information from videos,
including both visual frames and audio, up to 90 minutes long
*   Detect objects in an image and return bounding box coordinates for them

This tutorial demonstrates some possible ways to prompt the Gemini API with
images and video input, provides code examples,
and outlines prompting best practices with multimodal vision capabilities.
All output is text-only.

## Setup

Before you use the File API, you need to install the Gemini API SDK package and configure an API key. This section describes how to complete these setup steps.

### Install the Python SDK and import packages

The Python SDK for the Gemini API is contained in the [google-generativeai](https://pypi.org/project/google-generativeai/) package. Install the dependency using pip.

In [ ]:
!pip install -q -U google-generativeai

Import the necessary packages.

In [ ]:
import google.generativeai as genai
from IPython.display import Markdown

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
%cd Gemini_video

/content/drive/MyDrive/Gemini_video


In [ ]:
%ls

airplane_fly_1_body.mp4               hand_pass_1_body.mp4
airplane_offhand_1_body.mp4           headphones_offhand_1_body.mp4
airplane_pass_1_body.mp4              headphones_pass_1_body.mp4
airplane_pick_all_body.mp4            headphones_use_1_body.mp4
alarmclock_offhand_1_body.mp4         knife_pick_all_body.mp4
alarmclock_pass_1_body.mp4            lightbulb_pass_1_body.mp4
alarmclock_pick_all_body.mp4          mouse_offhand_1_body.mp4
alarmclock_see_1_body.mp4             mouse_pass_1_body.mp4
apple_eat_1_body.mp4                  mouse_pick_all_body.mp4
apple_offhand_1_body.mp4              mouse_use_1_body.mp4
apple_pass_1_body.mp4                 mug_drink_1_body.mp4
apple_pick_all_body.mp4               mug_drink_2_body.mp4
banana_eat_1_body.mp4                 mug_drink_3_body.mp4
banana_offhand_1_body.mp4             mug_drink_4_body.mp4
banana_pass_1_body.mp4                mug_offhand_1_body.mp4
banana_peel_1_body.mp4                mug_pass_1_body.mp4
banana_peel_2_body.

In [ ]:
%pwd

'/content/drive/MyDrive/Gemini_video'

### Set up your API key

The File API uses API keys for authentication and access. Uploaded files are associated with the project linked to the API key. Unlike other Gemini APIs that use API keys, your API key also grants access to data you've uploaded to the File API, so take extra care in keeping your API key secure. For more on keeping your keys
secure, see [Best practices for using API
keys](https://support.google.com/googleapi/answer/6310037).

Store your API key in a Colab Secret named `GOOGLE_API_KEY`. If you don't already have an API key, or are unfamiliar with Colab Secrets, refer to the [Authentication quickstart](https://github.com/google-gemini/gemini-api-cookbook/blob/main/quickstarts/Authentication.ipynb).

In [ ]:
from google.colab import userdata
GOOGLE_API_KEY=userdata.get('GOOGLE_API_KEY')

genai.configure(api_key=GOOGLE_API_KEY)

In [ ]:
model = genai.GenerativeModel("gemini-1.5-flash")
response = model.generate_content("Explain how AI works")
print(response.text)

Artificial intelligence (AI) is a broad field encompassing many techniques, but at its core, it's about creating systems that can perform tasks that typically require human intelligence.  These tasks include things like learning, problem-solving, decision-making, speech recognition, and visual perception.  There's no single "how it works" answer, as different AI approaches use different mechanisms.  However, here's a breakdown of some key concepts:

**1. Machine Learning (ML):** This is arguably the most prevalent approach to AI today.  Instead of explicitly programming rules, ML algorithms learn from data.  They identify patterns and relationships within the data to make predictions or decisions.  This learning process happens through various techniques:

* **Supervised Learning:** The algorithm is trained on a labeled dataset, meaning each data point is tagged with the correct answer.  For example, showing the algorithm pictures of cats and dogs, labeled accordingly, so it learns to 

## Prompting with images

In this tutorial, you will upload images using the File API or as inline data and generate content based on those images.

### Technical details (images)
Gemini 1.5 Pro and Flash support a maximum of 3,600 image files.

Images must be in one of the following image data [MIME types](https://developers.google.com/drive/api/guides/ref-export-formats):

-   PNG - `image/png`
-   JPEG - `image/jpeg`
-   WEBP - `image/webp`
-   HEIC - `image/heic`
-   HEIF - `image/heif`

Each image is equivalent to 258 tokens.

While there are no specific limits to the number of pixels in an image besides the model’s context window, larger images are scaled down to a maximum resolution of 3072x3072 while preserving their original aspect ratio, while smaller images are scaled up to 768x768 pixels. There is no cost reduction for images at lower sizes, other than bandwidth, or performance improvement for images at higher resolution.

For best results:

*   Rotate images to the correct orientation before uploading.
*   Avoid blurry images.
*   If using a single image, place the text prompt after the image.

## Image input

For total image payload size less than 20MB, it's recommended to either upload
base64 encoded images or directly upload locally stored image files.

### Base64 encoded images

You can upload public image URLs by encoding them as Base64 payloads.
You can use the httpx library to fetch the image URLs.
The following code example shows how to do this:

In [ ]:
import httpx
import base64

# Retrieve an image
image_path = "https://upload.wikimedia.org/wikipedia/commons/thumb/8/87/Palace_of_Westminster_from_the_dome_on_Methodist_Central_Hall.jpg/2560px-Palace_of_Westminster_from_the_dome_on_Methodist_Central_Hall.jpg"
image = httpx.get(image_path)

# Choose a Gemini model
model = genai.GenerativeModel(model_name="gemini-1.5-pro")

# Create a prompt
prompt = "Caption this image."
response = model.generate_content(
    [
        {
            "mime_type": "image/jpeg",
            "data": base64.b64encode(image.content).decode("utf-8"),
        },
        prompt,
    ]
)

Markdown(">" + response.text)

>A view of London, England showcases the Palace of Westminster (Houses of Parliament) and Big Ben with the London Eye and a cloudy sky in the background. Several other buildings, including part of St. Margaret's Church, and landmarks are visible in the sprawling cityscape. The River Thames snakes through the scene.

### Multiple images

To prompt with multiple images in Base64 encoded format, you can do the
following:

In [ ]:
import httpx
import base64

# Retrieve two images
image_path_1 = "https://upload.wikimedia.org/wikipedia/commons/thumb/8/87/Palace_of_Westminster_from_the_dome_on_Methodist_Central_Hall.jpg/2560px-Palace_of_Westminster_from_the_dome_on_Methodist_Central_Hall.jpg"
image_path_2 = "https://storage.googleapis.com/generativeai-downloads/images/jetpack.jpg"

image_1 = httpx.get(image_path_1)
image_2 = httpx.get(image_path_2)

# Create a prompt
prompt = "Generate a list of all the objects contained in both images."

response = model.generate_content([
{'mime_type':'image/jpeg', 'data': base64.b64encode(image_1.content).decode('utf-8')},
{'mime_type':'image/jpeg', 'data': base64.b64encode(image_2.content).decode('utf-8')}, prompt])

Markdown(response.text)

Here's a list of objects present in both images, keeping in mind that this is a somewhat whimsical pairing:


**Objects that are actually present:**

* **Buildings:**  Both images show numerous buildings. The first image features iconic London structures like the Houses of Parliament, Big Ben, and the London Eye, as well as many other buildings in the cityscape. The second image only features a drawing of a backpack.

**Objects that are conceptually implied/drawn:**

* **Transportation:** The first image depicts a view from above showing vehicles on the street below.  The second image depicts a backpack with fictional "retractable boosters", implying a mode of transportation, though not one present in the London skyline.

**Objects which share no direct relationship:**

The images have no objects in common aside from the broad category of "man-made structures".  The first shows a real-world cityscape; the second shows a conceptual drawing of a fictional backpack.


### Upload one or more locally stored image files

Alternatively, you can upload one or more locally stored image files..

You can download and use our drawings of [piranha-infested waters](https://storage.googleapis.com/generativeai-downloads/images/piranha.jpg) and a [firefighter with a cat](https://storage.googleapis.com/generativeai-downloads/images/firefighter.jpg). First, save these files to your local directory.

Then click **Files** on the left sidebar. For each file, click the **Upload** button, then navigate to that file's location and upload it:

<img width=400 src="https://ai.google.dev/tutorials/images/colab_upload.png">

When the combination of files and system instructions that you intend to send is larger than 20 MB in size, use the File API to upload those files. Smaller files can instead be called locally from the Gemini API:


In [ ]:
import PIL.Image

sample_file_2 = PIL.Image.open('piranha.jpg')
sample_file_3 = PIL.Image.open('firefighter.jpg')

In [ ]:
import google.generativeai as genai

# Choose a Gemini model.
model = genai.GenerativeModel(model_name="gemini-1.5-pro-latest")

# Create a prompt.
prompt = "Write an advertising jingle based on the items in both images."

response = model.generate_content([sample_file_2, sample_file_3, prompt])

Markdown(response.text)

Note that these inline data calls don't include many of the features available
through the File API, such as getting file metadata,
[listing](https://ai.google.dev/gemini-api/docs/vision?lang=python#list-files),
or [deleting files](https://ai.google.dev/gemini-api/docs/vision?lang=python#delete-files).

### Large image payloads

#### Upload an image file using the File API

When the combination of files and system instructions that you intend to send is larger than 20 MB in size, use the File API to upload those files.

**NOTE**: The File API lets you store up to 20 GB of files per project, with a per-file maximum size of 2 GB. Files are stored for 48 hours. They can be accessed in that period with your API key, but cannot be downloaded from the API. It is available at no cost in all regions where the Gemini API is available.

Upload the image using [`media.upload`](https://ai.google.dev/api/rest/v1beta/media/upload) and print the URI, which is used as a reference in Gemini API calls.

In [ ]:
!curl -o jetpack.jpg https://storage.googleapis.com/generativeai-downloads/images/jetpack.jpg

In [ ]:
# Upload the file and print a confirmation.
sample_file = genai.upload_file(path="jetpack.jpg",
                            display_name="Jetpack drawing")

print(f"Uploaded file '{sample_file.display_name}' as: {sample_file.uri}")

The `response` shows that the File API stored the specified `display_name` for the uploaded file and a `uri` to reference the file in Gemini API calls. Use `response` to track how uploaded files are mapped to URIs.

Depending on your use case, you can also store the URIs in structures such as a `dict` or a database.

#### Verify image file upload and get metadata

You can verify the API successfully stored the uploaded file and get its metadata by calling [`files.get`](https://ai.google.dev/api/rest/v1beta/files/get) through the SDK. Only the `name` (and by extension, the `uri`) are unique. Use `display_name` to identify files only if you manage uniqueness yourself.

In [ ]:
file = genai.get_file(name=sample_file.name)
print(f"Retrieved file '{file.display_name}' as: {sample_file.uri}")

Depending on your use case, you can store the URIs in structures, such as a `dict` or a database.

#### Prompt with the uploaded image and text

After uploading the file, you can make GenerateContent requests that reference the File API URI. Select the generative model and provide it with a text prompt and the uploaded image.

In [ ]:
# Choose a Gemini model.
model = genai.GenerativeModel(model_name="gemini-1.5-pro-latest")

# Prompt the model with text and the previously uploaded image.
response = model.generate_content([sample_file, "Describe how this product might be manufactured."])

Markdown(response.text)

## Capabilties

This section outlines specific vision capabilities of the Gemini model, including object detection and bounding box coordinates.

### Get bounding boxes

Gemini models are trained to return bounding box coordinates as relative widths or heights in the range of [0, 1]. These values are then scaled by 1000 and converted to integers. Effectively, the coordinates represent the bounding box on a 1000x1000 pixel version of the image. Therefore, you'll need to convert these coordinates back to the dimensions of your original image to accurately map the bounding boxes.

In [ ]:
# Choose a Gemini model.
model = genai.GenerativeModel(model_name="gemini-1.5-pro-latest")

# Create a prompt to detect bounding boxes.
prompt = "Return a bounding box for each of the objects in this image in [ymin, xmin, ymax, xmax] format."
response = model.generate_content([sample_file_2, prompt])

Markdown(response.text)

The model returns bounding box coordinates in the format
`[ymin, xmin, ymax, xmax]`. To convert these normalized coordinates
to the pixel coordinates of your original image, follow these steps:

1.    Divide each output coordinate by 1000.
1.    Multiply the x-coordinates by the original image width.
1.    Multiply the y-coordinates by the original image height.

To explore more detailed examples of generating bounding box coordinates and
visualizing them on images, review our [Object Detection cookbook example](https://github.com/google-gemini/cookbook/blob/main/examples/Object_detection.ipynb).

## Prompting with video

In this tutorial, you will upload a video using the File API and generate content based on those images.

### Technical details (video)

Gemini 1.5 Pro and Flash support up to approximately an hour of video data.

Video must be in one of the following video format [MIME types](https://developers.google.com/drive/api/guides/ref-export-formats):
  -   `video/mp4`
  -   `video/mpeg`
  -   `video/mov`
  -   `video/avi`
  -   `video/x-flv`
  -   `video/mpg`
  -   `video/webm`
  -   `video/wmv`
  -   `video/3gpp`

The File API service currently extracts image frames from videos at 1 frame per second (FPS) and audio at 1Kbps, single channel, adding timestamps every second. These rates are subject to change in the future for improvements in inference.

**NOTE:** The finer details of fast action sequences may be lost at the 1 FPS frame sampling rate. Consider slowing down high-speed clips for improved inference quality.

Individual frames are 258 tokens, and audio is 32 tokens per second. With metadata, each second of video becomes ~300 tokens, which means a 1M context window can fit slightly less than an hour of video.

To ask questions about time-stamped locations, use the format `MM:SS`, where the first two digits represent minutes and the last two digits represent seconds.

For best results:

*   Use one video per prompt.
*   If using a single video, place the text prompt after the video.

### Upload a video file to the File API

**NOTE**: The File API lets you store up to 20 GB of files per project, with a per-file maximum size of 2 GB. Files are stored for 48 hours. They can be accessed in that period with your API key, but they cannot be downloaded using any API. It is available at no cost in all regions where the Gemini API is available.

The File API accepts video file formats directly. This example uses the short NASA film ["Jupiter's Great Red Spot Shrinks and Grows"](https://www.youtube.com/watch?v=JDi4IdtvDVE0). Credit: Goddard Space Flight Center (GSFC)/David Ladd (2018).

> "Jupiter's Great Red Spot Shrinks and Grows" is in the public domain and does not show identifiable people. ([NASA image and media usage guidelines.](https://www.nasa.gov/nasa-brand-center/images-and-media/))

Start by retrieving the short video:

In [ ]:
!wget https://storage.googleapis.com/generativeai-downloads/images/GreatRedSpot.mp4

--2025-01-20 06:13:34--  https://storage.googleapis.com/generativeai-downloads/images/GreatRedSpot.mp4
Resolving storage.googleapis.com (storage.googleapis.com)... 142.250.157.207, 142.251.8.207, 142.251.170.207, ...
Connecting to storage.googleapis.com (storage.googleapis.com)|142.250.157.207|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 238090979 (227M) [video/mp4]
Saving to: ‘GreatRedSpot.mp4’

GreatRedSpot.mp4    100%[===================>] 227.06M  24.7MB/s    in 11s     

2025-01-20 06:13:46 (19.8 MB/s) - ‘GreatRedSpot.mp4’ saved [238090979/238090979]



Upload the video to the File API and print the URI.

In [ ]:
%ls


airplane_fly_1_body.mp4               cubesmall_inspect_1_body.mp4
airplane_offhand_1_body.mp4           cubesmall_offhand_1_body.mp4
airplane_pass_1_body.mp4              cubesmall_pass_1_body.mp4
airplane_pick_all_body.mp4            cubesmall_pick_all_body.mp4
alarmclock_offhand_1_body.mp4         cup_drink_1_body.mp4
alarmclock_pass_1_body.mp4            cup_drink_2_body.mp4
alarmclock_pick_all_body.mp4          cup_pass_1_body.mp4
alarmclock_see_1_body.mp4             cup_pick_all_body.mp4
apple_eat_1_body.mp4                  cup_pour_1_body.mp4
apple_offhand_1_body.mp4              cylinderlarge_inspect_1_body.mp4
apple_pass_1_body.mp4                 cylinderlarge_offhand_1_body.mp4
apple_pick_all_body.mp4               cylinderlarge_pass_1_body.mp4
banana_eat_1_body.mp4                 cylinderlarge_pick_all_body.mp4
banana_offhand_1_body.mp4             cylindermedium_inspect_1_body.mp4
banana_pass_1_body.mp4                cylindermedium_offhand_1_body.mp4
banana_peel_1_body

In [ ]:
video_file_name = "GreatRedSpot.mp4"

print(f"Uploading file...")
video_file = genai.upload_file(path=video_file_name)
print(f"Completed upload: {video_file.uri}")

Uploading file...
Completed upload: https://generativelanguage.googleapis.com/v1beta/files/xqble611y36s


### Verify file upload and check state

Verify the API has successfully received the files by calling the [`files.get`](https://ai.google.dev/api/rest/v1beta/files/get) method.

**NOTE**: Video files have a `State` field in the File API. When a video is uploaded, it will be in the `PROCESSING` state until it is ready for inference. Only `ACTIVE` files can be used for model inference.

In [ ]:
import time

# Check whether the file is ready to be used.
while video_file.state.name == "PROCESSING":
    print('.', end='')
    time.sleep(10)
    video_file = genai.get_file(video_file.name)

if video_file.state.name == "FAILED":
  raise ValueError(video_file.state.name)

.

In [32]:
## 我的
video_file_name = "apple_eat_1_body.mp4"
video_file = genai.upload_file(path=video_file_name)

import time
# Check whether the file is ready to be used.
while video_file.state.name == "PROCESSING":
    print('.', end='')
    time.sleep(10)
    video_file = genai.get_file(video_file.name)
if video_file.state.name == "FAILED":
  raise ValueError(video_file.state.name)

# Create the prompt.
# prompt = "Summarize this video. Then create a quiz with answer key based on the information in the video. Describe the video in the first person with the people in the video, and focus on the interactions of the characters."
# prompt = "Summarize this video. And please generate a description similar to the following: 'Grab the apple with the right hand and eat it.' And please use the person in the video as the first person to describe it. And the wireframe model in the video is a human. And in the description, please only appear human and objects that interact with human, and do not appear unrelated objects."
prompt = "Summarize this video. And please generate a description similar to the following: 'Grab the apple with the right hand and eat it.' And please use the person in the video as the first person to describe it. And the wireframe model in the video is a human. And the description focuses only on human and objects that interact with human."
# Choose a Gemini model.
# model = genai.GenerativeModel(model_name="gemini-1.5-pro-latest")
model = genai.GenerativeModel(model_name="gemini-1.5-pro")
# Make the LLM request.
print("Making LLM inference request...")
response = model.generate_content([video_file, prompt],request_options={"timeout": 600})
# Print the response, rendering any Markdown
Markdown(response.text)

.

KeyboardInterrupt: 

In [ ]:
def split_and_filter(input_string):
    # 将字符串按照 "_" 分割
    parts = input_string.split('_')

    # 过滤掉为 "1", "2", "3", "4" 的字符串
    filtered_parts = [part for part in parts if part not in ['1', '2', '3', '4']]

    return filtered_parts

In [33]:
import time
def get_description(video_file_name):
  # video_file_name = "apple_eat_1_body.mp4"
  video_file = genai.upload_file(path=video_file_name)

  # Check whether the file is ready to be used.
  while video_file.state.name == "PROCESSING":
    print('.', end='')
    time.sleep(10)
    video_file = genai.get_file(video_file.name)
  if video_file.state.name == "FAILED":
    raise ValueError(video_file.state.name)

  result=split_and_filter(video_file_name)
  print("hint："+result[0]+"、"+result[1])

  # Create the prompt.
  # prompt = "Summarize this video. Then create a quiz with answer key based on the information in the video. Describe the video in the first person with the people in the video, and focus on the interactions of the characters."
  # prompt = "Summarize this video. And please generate a description similar to the following: 'Grab the apple with the right hand and eat it.' And please use the person in the video as the first person to describe it. And the wireframe model in the video is a human. And in the description, please only appear human and objects that interact with human, and do not appear unrelated objects."
  # prompt = "Summarize this video. And please generate a description similar to the following: 'Grab the apple with the right hand and eat it.' And please use the person in the video as the first person to describe it. And the wireframe model in the video is a human. And the description focuses only on human and objects that interact with human. And here is a hint:"+result[0]+result[1]
  prompt = "Summarize this video. And please generate a description similar to the following: 'Grab the apple with the right hand and eat it.' And please use the person in the video as the first person to describe it. And the wireframe model in the video is a human. And the description focuses only on human and objects that interact with human. And here is a hint:"+result[0]+result[1] + \
         "Watch this video, identify the actions and devise a plan using chain-of-thought. Extract detailed actions using this schema:\n" + \
         "Task: {\"task description\"}\n" + \
         "Plan: {\"plan with chain-of-thought\"} Actions: {{" + \
         "\"number\"}: {'verb'}{'noun'}}.\n" + \
         "pre-action: [Description]\n" + \
         "action: [Description]\n" + \
         "post-action: [Description]\n" + \
         "Merge descriptions: Merge the three-part descriptions:pre-action,action,post-action"
  # prompt = "Watch this video, identify the actions and devise a plan using chain-of-thought. Extract detailed actions using this schema:\n" + \
  #        "Task: {\"task description\"}\n" + \
  #        "Plan: {\"plan with chain-of-thought\"} Actions: {{" + \
  #        "\"number\"}: {'verb'}{'noun'}}.\n" + \
  #        "pre-action: [Description]\n" + \
  #        "action: [Description]\n" + \
  #        "post-action: [Description]\n" + \
  #        "Summarize this video. And please generate a description similar to the following: 'Grab the apple with the right hand and eat it.' And please use the person in the video as the first person to describe it. And the wireframe model in the video is a human. And the description focuses only on human and objects that interact with human. And here is a hint:"+result[0]+result[1]


  # print(prompt)

  # Choose a Gemini model.
  # model = genai.GenerativeModel(model_name="gemini-1.5-pro-latest")
  model = genai.GenerativeModel(model_name="gemini-1.5-pro")
  # Make the LLM request.
  print("Making LLM inference request...")
  response = model.generate_content([video_file, prompt],request_options={"timeout": 600})
  # Print the response, rendering any Markdown
  # Markdown(response.text)
  display(Markdown(response.text))

In [ ]:
%pwd

'/content/drive/MyDrive/Gemini_video'

In [ ]:
from google.colab import files
import os
from google.api_core.exceptions import TooManyRequests

# 获取上传的文件名
def get_mp4_files(directory):
    # 获取文件夹中的所有文件
    files = os.listdir(directory)

    # 筛选出所有以 .mp4 结尾的文件
    mp4_files = [file for file in files if file.lower().endswith('.mp4')]
    mp4_files.sort()
    return mp4_files

# 设置 Colab 环境中存储文件的目录
folder_path = '.'

# 获取 MP4 文件列表
mp4_files = get_mp4_files(folder_path)

# 打印所有 MP4 文件名
for file in mp4_files:
    print(file)
    retries=10
    wait_time=1
    for attempt in range(retries):
        try:
            get_description(file)  # 调用获取描述的函数
            break  # 如果请求成功，则退出循环
        except TooManyRequests:
            if attempt < retries - 1:
                print(f"超出请求限制，{wait_time}秒后重试...")
                time.sleep(wait_time)  # 等待后重试
            else:
                print("已达到最大重试次数，请稍后再试。")
    # get_description(file)

airplane_fly_1_body.mp4
.hint：airplane、fly
Making LLM inference request...


超出请求限制，1秒后重试...
hint：airplane、fly
Making LLM inference request...


超出请求限制，1秒后重试...
.hint：airplane、fly
Making LLM inference request...


超出请求限制，1秒后重试...
.hint：airplane、fly
Making LLM inference request...


超出请求限制，1秒后重试...
.hint：airplane、fly
Making LLM inference request...


超出请求限制，1秒后重试...


### Prompt with a video and text

Once the uploaded video is in the `ACTIVE` state, you can make `GenerateContent` requests that specify the File API URI for that video. Select the generative model and provide it with the uploaded video and a text prompt.

In [ ]:
# Create the prompt.
prompt = "Summarize this video. Then create a quiz with answer key based on the information in the video."

# Choose a Gemini model.
model = genai.GenerativeModel(model_name="gemini-1.5-pro-latest")

# Make the LLM request.
print("Making LLM inference request...")
response = model.generate_content([video_file, prompt],
                                  request_options={"timeout": 600})

# Print the response, rendering any Markdown
Markdown(response.text)

Making LLM inference request...


The video highlights some facts about Jupiter, primarily about the shrinking size of the Great Red Spot.  

Here is a quiz about the video:

1. Jupiter is the __________ and oldest planet in our solar system.
a) smallest
b) largest
c) second largest

2. What type of planet is Jupiter?
a) a gas giant
b) a terrestrial planet
c) an ice giant

3. What is the Great Red Spot?
a) a large mountain
b) a giant storm
c) Jupiter’s largest moon

4. How long has the Great Red Spot lasted?
a) over a century
b) over two centuries
c) over three centuries

5. Over time, the Great Red Spot has been:
a) shrinking and becoming more oval
b) growing and becoming more oval
c) shrinking and becoming rounder

6. The color of the Great Red Spot is:
a) fading
b) intensifying
c) becoming more yellow


Answer Key:
1. b
2. a
3. b
4. a
5. c
6. b




### Refer to timestamps in the content

You can use timestamps of the form `MM:SS` to refer to specific moments in the video.

In [ ]:
# Create the prompt.
prompt = "What are the examples given at 01:05 and 01:19 supposed to show us?"

# Choose a Gemini model.
model = genai.GenerativeModel(model_name="gemini-1.5-pro-latest")

# Make the LLM request.
print("Making LLM inference request...")
response = model.generate_content([prompt, video_file],
                                  request_options={"timeout": 600})
Markdown(response.text)

Making LLM inference request...


The video shows us the examples of an ice skater and a potter to help visualize the principles behind the changes to Jupiter’s Great Red Spot.

At [00:00:05], we see an ice skater spinning on one leg. She initially spins slowly with arms outstretched and then pulls them in, which increases her rotational velocity. This is analogous to how it was thought that the Great Red Spot might speed up as it shrunk in width. However, that was later shown not to be the case.

At [00:00:19], a potter is shown shaping a spinning mound of clay on a wheel. As the clay spins, the potter’s hands mold it into a taller, thinner form. This is what is actually happening to the Great Red Spot: the storm shrinks from our perspective, but it becomes taller.

### Transcribe video and provide visual descriptions

The Gemini models can transcribe and provide visual descriptions of video content
by processing both the audio track and visual frames.
For visual descriptions, the model samples the video at a rate of **1 frame
per second**. This sampling rate may affect the level of detail in the
descriptions, particularly for videos with rapidly changing visuals.

In [ ]:
# Create the prompt.
prompt = "Transcribe the audio from this video, giving timestamps for salient events in the video. Also provide visual descriptions."

# Choose a Gemini model.
model = genai.GenerativeModel(model_name="gemini-1.5-pro-latest")

# Make the LLM request.
print("Making LLM inference request...")
response = model.generate_content([video_file, prompt],
                                  request_options={"timeout": 600})
Markdown(response.text)

Making LLM inference request...


Okay, here’s your transcription with salient timestamps and visual descriptions:

[00:00:00] Jupiter is shown against the backdrop of the Milky Way.

[00:00:15] Jupiter’s rings become visible as the planet moves closer to the camera, then rotates so that it fills the screen. The Great Red Spot is visible on the planet’s surface.

[00:00:20] Jupiter is centered on a black background.

[00:00:26] A zoom into the Great Red Spot takes place, accompanied by a fade transition to [00:00:32] a close-up of the spot, which fills the screen. A moon and its shadow are visible moving across the scene.

[00:00:41] Zoom out transition back to Jupiter centered against the black background.

[00:00:49] Inset: Images of Jupiter show the shrinking and darkening of the Great Red Spot over time, from 1995 to 2015.

[00:00:59] Zoom transition to a close-up of the Great Red Spot, with the lower edge of the image now visible above the spot’s swirling cloud patterns.

[00:01:04] A black-and-white clip shows an ice skater spinning slowly and then pulling in her arms to increase her speed.

[00:01:12] Zoom transition to the Great Red Spot.

[00:01:17] Split screen. Left: A graph shows the height of the Great Red Spot increasing from 2014 to 2017. Right: A black-and-white clip shows hands molding a pot to a taller shape on a pottery wheel.

[00:01:25] Images of the Great Red Spot from 1995 to 2015 show how the storm has shrunk to a smaller and rounder shape.

[00:01:31] Earth is placed in the image to show how the Great Red Spot used to be big enough to contain three Earths.

[00:01:36] The Voyager 2 space probe flies close to Jupiter and its moon, Io.

[00:01:44] The Juno space probe is seen in the foreground of Jupiter as it orbits the planet.

[00:01:47] Jupiter, centered on the black background of space and the Milky Way.

[00:01:54] The Great Red Spot fills the screen.

[00:01:58] The Juno space probe is seen with the Earth in the background. The NASA logo and Goddard Space Flight Center appear.


Hope this helps!

## List files

You can list all uploaded files and their URIs using `files.list_files()`.

In [ ]:
# List all files
for file in genai.list_files():
    print(f"{file.display_name}, URI: {file.uri}")

GreatRedSpot.mp4, URI: https://generativelanguage.googleapis.com/v1beta/files/xqble611y36s


## Delete files

Files are automatically deleted after 2 days. You can also manually delete them using `files.delete()`.

In [ ]:
genai.delete_file(video_file.name)
print(f'Deleted file {video_file.uri}')